# Minimind整体架构

```text
训练/推理命令参数
        ↓
MiniMindConfig（确定模型规格）
        ↓
MiniMindForCausalLM（语言模型外壳）
        ↓
MiniMindModel
        ↓
Embedding → 8个 Transformer Block → RMSNorm → LM Head
                       ↓
              Attention + FFN/MoE
```

# 1. 核心模块详解
MiniMind 由多个高度模块化的组件构成，以下再次总结各核心类的功能与设计原理。

## 1.1 基础组件

### 1.1.1 **MiniMindConfig (`PretrainedConfig`)**
  - **作用**：MiniMindConfig 是整个 MiniMind 模型的“配置总表”或“设计蓝图”。
它本身不执行神经网络计算，而是统一规定模型应该有多大、使用什么结构、支持多长上下文，以及是否启用 MoE。

后面的模型构建、训练、推理和权重保存都会读取这里的参数。

In [ ]:

# 🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏
#                                     MiniMind Config
# 🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏

class MiniMindConfig(PretrainedConfig):
    """集中保存 MiniMind 的模型结构、注意力、位置编码和 MoE 配置。"""

    # Hugging Face 用于识别模型类型的名称
    model_type = "minimind"

    def __init__(self, hidden_size=768, num_hidden_layers=8, use_moe=False, **kwargs):
        # 初始化 Hugging Face 配置基类，使配置支持保存和加载
        super().__init__(**kwargs)

        # 模型主体配置
        self.hidden_size = hidden_size              # Token 隐藏向量维度，也是模型主干宽度
        self.num_hidden_layers = num_hidden_layers  # Transformer Block 的堆叠层数
        self.use_moe = use_moe                      # 是否使用 MoE 替换普通前馈网络
        self.dropout = kwargs.get("dropout", 0.0)   # Dropout 概率，0 表示默认关闭

        # 词表与特殊 Token
        self.vocab_size = kwargs.get("vocab_size", 6400)  # 词表大小，决定 Embedding 和 LM Head 大小
        self.bos_token_id = kwargs.get("bos_token_id", 1) # 序列开始 Token
        self.eos_token_id = kwargs.get("eos_token_id", 2) # 序列结束 Token

        # 注意力配置
        self.flash_attn = kwargs.get("flash_attn", True)                 # 是否优先使用高效注意力
        self.num_attention_heads = kwargs.get("num_attention_heads", 8)  # Query 注意力头数量
        self.num_key_value_heads = kwargs.get("num_key_value_heads", 4)  # Key/Value 头数量，少于 Q 头时构成 GQA
        self.head_dim = kwargs.get("head_dim", self.hidden_size // self.num_attention_heads)  # 单个注意力头维度，默认 96

        # 前馈网络配置
        self.hidden_act = kwargs.get("hidden_act", "silu")  # SwiGLU 使用的激活函数
        self.intermediate_size = kwargs.get("intermediate_size", math.ceil(hidden_size * math.pi / 64) * 64)  # FFN 中间层维度，默认 2432

        # 位置编码与归一化
        self.max_position_embeddings = kwargs.get("max_position_embeddings", 32768)  # 最大位置编码长度
        self.rms_norm_eps = kwargs.get("rms_norm_eps", 1e-6)  # RMSNorm 的数值稳定项
        self.rope_theta = kwargs.get("rope_theta", 1e6)       # RoPE 位置编码的频率基数

        # 输入 Embedding 与输出 LM Head 是否共享权重
        self.tie_word_embeddings = kwargs.get("tie_word_embeddings", True)

        # 是否在推理时使用 YaRN 进行长上下文外推
        self.inference_rope_scaling = kwargs.get("inference_rope_scaling", False)

        # 开启后将原始 2048 上下文按 16 倍外推至约 32768
        self.rope_scaling = {
            "beta_fast": 32,                            # 高频分量校正边界
            "beta_slow": 1,                             # 低频分量校正边界
            "factor": 16,                               # 上下文外推倍数
            "original_max_position_embeddings": 2048,   # 训练时原始上下文长度
            "attention_factor": 1.0,                    # 注意力缩放系数
            "type": "yarn"                              # 使用 YaRN 外推算法
        } if self.inference_rope_scaling else None

        # MoE 专用配置，use_moe=False 时不会参与模型计算
        self.num_experts = kwargs.get("num_experts", 4)  # 每层的专家总数
        self.num_experts_per_tok = kwargs.get("num_experts_per_tok", 1)  # 每个 Token 选择的专家数量
        self.moe_intermediate_size = kwargs.get("moe_intermediate_size", self.intermediate_size)  # 单个专家的 FFN 中间维度
        self.norm_topk_prob = kwargs.get("norm_topk_prob", True)  # 是否归一化所选专家的路由权重
        self.router_aux_loss_coef = kwargs.get("router_aux_loss_coef", 5e-4)  # 专家负载均衡辅助损失系数

### 1.1.2 RMSNorm (`Root Mean Square Layer Normalization`)
  - **作用**：层归一化。
  - **原理**：相比标准的 LayerNorm，RMSNorm 移除了中心化操作（不减均值），仅保留缩放。
  - **优势**：计算量更小，数值稳定性更好，是目前主流大模型（如 LLaMA, Gemma）的标准配置。

In [ ]:

# 🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏
#                                     MiniMind Model
# 🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏🌎🌍🌏
class RMSNorm(torch.nn.Module):
    """
    RMSNorm (Root Mean Square Layer Normalization)
    
    RMSNorm 是 LayerNorm 的简化版本，只对输入进行缩放，不进行中心化（不减去均值）。
    相比 LayerNorm，RMSNorm 计算更高效，且在实际应用中效果相当。
    
    公式：
        RMSNorm(x) = weight * (x / sqrt(mean(x^2) + eps))
    
    与 LayerNorm 的区别：
        - LayerNorm: (x - mean(x)) / sqrt(var(x) + eps)
        - RMSNorm: x / sqrt(mean(x^2) + eps)
        - RMSNorm 不减去均值，只进行缩放，计算更快
    """
    def __init__(self, dim: int, eps: float = 1e-5):
        """
        初始化 RMSNorm 层
        Args:
            dim: 输入特征的维度
            eps: 防止除零的小常数（默认 1e-5）
        """
        super().__init__()
        self.eps = eps 
        self.weight = nn.Parameter(torch.ones(dim))
        # 可学习的缩放参数，初始化为全 1

    def norm(self, x):
        """
        计算 RMS 归一化
        公式：x / sqrt(mean(x^2) + eps)
        Args:
            x: 输入张量 [..., dim]
        Returns:
            归一化后的张量，形状与输入相同
        """
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        # x.pow(2): 计算 x 的平方
        # .mean(-1, keepdim=True): 在最后一个维度上求均值，保持维度
        # torch.rsqrt: 计算 1/sqrt，比先 sqrt 再除更快

    def forward(self, x):
        """
        前向传播：
        Args:
            x: 输入张量，可以是任意精度（float16, bfloat16, float32）
        Returns:
            归一化并缩放后的张量，保持原始精度
        """
        return (self.weight * self.norm(x.float())).type_as(x)
        # 先转换为 float32 进行归一化计算（提高数值稳定性）
        # 然后转换回原始精度（type_as(x)）
        # 最后乘以可学习的权重

## 1.2 注意力机制 (Attention)

- **Attention 类**

  Attention 类实现了 MiniMind 的因果自注意力机制，主要包含：
  - 将输入映射为 Query、Key、Value；
  - 使用 GQA 减少 KV Head 数量；
  - 对 Query、Key 应用 RMSNorm 和 RoPE；
  - 在推理时复用 KV Cache；
  - 优先使用高效注意力，条件不满足时使用手写注意力；
  - 输出融合上下文后的隐藏状态。

需要注意：该类只计算注意力，不负责残差连接；残差连接在 MiniMindBlock 中完成。

这里的代码实现在`03-大模型优化方法`中

## 1.3 前馈网络 (FeedForward & MoE)

这是 MiniMind 区别于普通 Transformer 的核心区域，支持两种模式：

### 1.3.1 FeedForward (标准 Dense 模式)
  - **架构**：采用 **SwiGLU** 激活函数。
  - **流程**：输入经过门控投影（Gate）和上投影（Up），相乘后过 Swish 激活，最后经下投影（Down）输出。
  - **优势**：相比传统的 ReLU+FFN，SwiGLU 具有更好的收敛性和性能。

In [ ]:
class FeedForward(nn.Module):
    """
    SwiGLU 前馈网络
    
    实现了 SwiGLU (Swish-Gated Linear Unit) 激活函数的前馈网络。
    SwiGLU 是 GLU (Gated Linear Unit) 的变体，使用 Swish/SiLU 作为门控激活函数。
    
    公式：
        FFN(x) = down_proj(Swish(gate_proj(x)) * up_proj(x))
    
    其中：
        - gate_proj: 门控投影，用于生成门控信号
        - up_proj: 上投影，用于生成特征
        - Swish(x) = x * sigmoid(x) = x * silu(x)
        - down_proj: 下投影，将中间维度映射回 hidden_size
    
    相比标准 FFN (ReLU(xW1)W2)，SwiGLU 通常有更好的性能。
    """
    def __init__(self, config: MiniMindConfig, intermediate_size: int = None):
        super().__init__()
        intermediate_size = intermediate_size or config.intermediate_size
        self.gate_proj = nn.Linear(config.hidden_size, intermediate_size, bias=False)
        # gate_proj: 门控投影，hidden_size -> intermediate_size
        self.down_proj = nn.Linear(intermediate_size, config.hidden_size, bias=False)
        # down_proj: 下投影，intermediate_size -> hidden_size
        self.up_proj = nn.Linear(config.hidden_size, intermediate_size, bias=False)
        # up_proj: 上投影，hidden_size -> intermediate_size
        self.act_fn = ACT2FN[config.hidden_act]
        # 激活函数：通常是 'silu' (Swish)

    def forward(self, x):
        """
        前向传播
        SwiGLU 公式：FFN(x) = down_proj(Swish(gate_proj(x)) * up_proj(x))
        
        Args:
            x: 输入张量 [batch, seq_len, hidden_size]  
        Returns:
            输出张量 [batch, seq_len, hidden_size]
        """
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))
        #计算并返回 FFN(x) = down_proj(Swish(gate_proj(x)) * up_proj(x))


# 1.3.2 MOEFeedForward (混合专家模式)
  - **架构**：DeepSeek-V2/V3 风格的 MoE 架构。
  - **组成**：
    1. **Shared Experts (共享专家)**：`n_shared_experts` 个，处理**所有** Token，旨在捕获通用知识。
    2. **Routed Experts (路由专家)**：`n_routed_experts` 个，由门控网络动态选择。
    3. **MoEGate (门控)**：计算每个 Token 对各个专家的评分，选出 Top-K 个专家，并计算**辅助损失 (Aux Loss)** 以防止负载不均。
  - **流程**：`Output = 共享专家输出 + Σ(路由专家输出 * 门控权重)`。

In [ ]:
class MOEFeedForward(nn.Module):
    def __init__(self, config: MiniMindConfig):
        super().__init__()
        self.config = config # 保存 MoE 需要的配置参数
        self.gate = nn.Linear(config.hidden_size, config.num_experts, bias=False)
        #  gate: 路由/门控投影，hidden_size -> num_experts 为每个 token 选择专家并计算权重
        self.experts = nn.ModuleList([ 
            FeedForward(
                config, 
                intermediate_size=config.moe_intermediate_size
            ) 
            for _ in range(config.num_experts)
        ]) # 创建 num_experts 个相互独立的专家
        self.act_fn = ACT2FN[config.hidden_act]
        # 根据配置取得激活函数

    def forward(self, x):
        batch_size, seq_len, hidden_dim = x.shape
        x_flat = x.view(-1, hidden_dim) # [B,S,H]-->[B * S,H]记为[N,H]
        # 将x展开，所有token数=batch_size * seq_len，等价于x.reshape(-1, hidden_dim)
        scores = F.softmax(self.gate(x_flat), dim=-1)
        # 计算所有专家的路由概率
        # 先用self.gate(x_flat)得到每个token对每个专家的logits
        # 然后用softmax得到概率分布，[N,H]->[N,E]
        
        topk_weight, topk_idx = torch.topk(
            scores, 
            k=self.config.num_experts_per_tok, 
            dim=-1, 
            sorted=False
        )# 对每个 Token 选择概率最大的K个专家[N,K]

        if self.config.norm_topk_prob: 
            topk_weight = topk_weight / (
                topk_weight.sum(dim=-1, keepdim=True) + 1e-20
            ) # 对选中的K个专家的权重进行归一化，确保它们的和为1 [N,K]

        y = torch.zeros_like(x_flat)
        #创建与 x_flat 相同形状的零张量：y([N,H])，后续每个专家的加权结果都会累加到这里。
        
        #逐专家执行稀疏计算
        #代码虽然遍历了所有专家，但每个专家只计算分配给自己的 Token，因此专家内部计算是稀疏的
        for i, expert in enumerate(self.experts):
            mask = (topk_idx == i)
            #mask是一个布尔张量，形状为[N,K]，表示每个Token选择的K个专家中是否有当前专家i

            if mask.any():
            # 如果至少有一个 True，说明当前专家收到了至少一个 Token，需要执行计算
                token_idx = mask.any(dim=-1).nonzero().flatten()
                #mask.any(dim=-1)：[N,K]->[N]，判断每个 Token 是否选择了当前专家
                #.nonzero():找到为真的位置; .flatten():将1为真的位置整理为一维索引[N]->[N_i]
                weight = topk_weight[mask].view(-1, 1)
                #使用同一个布尔掩码提取专家i对应所有选择它的token的权重[N_i]

                #执行专家并累加输出
                y.index_add_(
                    0, 
                    token_idx, 
                    (expert(x_flat[token_idx]) * weight).to(y.dtype)
                )#x_flat[token_idx]:取出当前专家i负责的 Token :[N_i,H]
                 #expert(.):送入对应的 SwiGLU Expert_i
                 #乘路由权重weight,累加回原 Token 位置
                 # (使用“加法”而不是直接赋值，是因为一个 Token 可能被多个专家处理)
            elif self.training:
                y[0, 0] += 0 * sum(p.sum() for p in expert.parameters())
            #当某个专家没有收到任何 Token 时，进行0计算，生成网络连接

        #负载均衡辅助损失
        #只在训练时计算：
        if self.training and self.config.router_aux_loss_coef > 0:
            load = F.one_hot(
                topk_idx, 
                self.config.num_experts
            ).float().mean(0)
            #对topk_idx的token维求平均
            #得到laod，laod_i：(选择专家i的token数量)/N
            self.aux_loss = (
                load * scores.mean(0)
            ).sum() * self.config.num_experts * self.config.router_aux_loss_coef
            #scores.mean(0):计算所有 Token 对每个专家的平均路由概率，得到：importance_i
            #router_aux_loss_coef: 负载均衡损失 = [sum_i (load_i * importance_i)] * num_experts * router_aux_loss_coef
            #路由负载均衡辅助损失的权重系数
        else:
        #推理时不需要负载均衡损失，因为推理不进行反向传播。
            self.aux_loss = scores.new_zeros(1).squeeze()
            #推理时设为零
        
        #将展开的 Token 重新恢复为序列结构
        return y.view(batch_size, seq_len, hidden_dim)

## 1.4 模型骨架

### 1.4.1 MiniMindBlock

**单个 Transformer Block，完成注意力、前馈网络和残差连接**

 MiniMindBlock 表示单个 Transformer 层，采用 Pre-Norm 结构：
 ```text
 输入
 → RMSNorm
 → Self-Attention
 → 残差连接
 → RMSNorm
 → FFN / MoE
 → 残差连接
 → 输出
 ```

> 这个部分是模型拼接，将GQA层和FFN层拼接在一起

In [ ]:
class MiniMindBlock(nn.Module):
    def __init__(self, layer_id: int, config):
        super().__init__()
        # 基础配置
        self.num_attention_heads = config.num_attention_heads
        self.hidden_size = config.hidden_size
        self.head_dim = config.hidden_size // config.num_attention_heads

        # 自注意力模块，内部实现了 RoPE 相对位置编码
        self.self_attn = Attention(config)

        # 当前 Block 的层编号（可用于层内权重共享、分层控制等）
        self.layer_id = layer_id
        # Attention 前的 RMSNorm
        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        # Feed Forward 前的 RMSNorm
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        # 前馈网络模块，可配置是否使用专家混合（MoE）
        self.mlp = FeedForward(config) if not config.use_moe else MOEFeedForward(config)

    def forward(
        self,
        hidden_states,          # 输入的隐藏状态 [batch_size, seq_len, hidden_dim]
        position_embeddings,    # RoPE 位置编码 [seq_len, head_dim]
        past_key_value=None,    # KV 缓存 (key, value)，用于加速推理
        use_cache=False,        # 是否缓存当前层的 KV
        attention_mask=None     # attention 掩码
    ):
        # ---------------------- Self-Attention 层 ----------------------
        residual = hidden_states  # 保存残差连接

        # 对输入做 RMSNorm，再送入自注意力层
        hidden_states, present_key_value = self.self_attn(
            self.input_layernorm(hidden_states),  # LayerNorm 后输入 attention
            position_embeddings,                 # Rotary PE 传入 Attention
            past_key_value,                      # 过往 KV 缓存（一般在推理阶段使用）
            use_cache,                           # 是否缓存当前层 KV（一般在推理阶段使用）
            attention_mask                       # 注意力掩码（padding token不计算注意力矩阵）
        )

        # 残差连接：原始输入 + attention 输出
        hidden_states += residual

        # ---------------------- MLP 层 ----------------------
        # MLP 前再做一次 RMSNorm
        normed_hidden = self.post_attention_layernorm(hidden_states)

        # 残差连接：再加上 MLP 的输出
        hidden_states = hidden_states + self.mlp(normed_hidden)

        # 返回新的 hidden_states 和 当前层的 KV 缓存
        return hidden_states, present_key_value

# 2. 整体架构组装
```text
MiniMindForCausalLM
    └── MiniMindModel
          └── 多个 MiniMindBlock
                ├── Attention
                └── FeedForward / MOEFeedForward
```


## 2.1 整体架构组装原理
MiniMind 采用 Decoder-Only Transformer 和自回归语言建模方式，其数据流如下。


**1 输入与词嵌入**
- 输入的 Token IDs input_ids 首先进入 MiniMindModel。
- embed_tokens 将离散的 Token ID 映射为 hidden_size 维的隐藏向量。
- 随后对隐藏向量应用 Dropout。

**2 位置编码准备**
- 模型使用 RoPE 旋转位置编码。
- freqs_cos 和 freqs_sin 通常在模型初始化时预计算。
- 前向传播时，根据 KV Cache 长度 start_pos 和当前序列长度取出所需的位置编码。
- 如果启用 rope_scaling，则通过 YaRN 对 RoPE 进行长上下文外推。

**3 Transformer 层堆叠**

Hidden States 依次经过 num_hidden_layers 个 MiniMindBlock。每层执行：
```text
RMSNorm
    ↓
Self-Attention + RoPE
    ↓
第一次残差连接
    ↓
RMSNorm
    ↓
FeedForward / MOEFeedForward
    ↓
第二次残差连接
```
- Attention 负责聚合上下文信息。
- 推理时可以使用 KV Cache 复用历史 Key 和 Value。
- use_moe=False 时使用普通 SwiGLU 前馈网络。
- use_moe=True 时使用 MoE，并计算专家路由及负载均衡辅助损失。

**4 主干网络输出**
- 所有 Transformer 层执行完成后，再经过一次最终的 RMSNorm。
- MiniMindModel 返回最终隐藏状态、各层 KV Cache 和 MoE 辅助损失。

**5 词表预测与训练损失**
- MiniMindForCausalLM 使用 lm_head 将隐藏状态从 hidden_size 映射到 vocab_size，得到词表 Logits。
- 训练时使用当前位置预测下一个 Token，即比较：
```text 
logits[:, :-1]  ↔  labels[:, 1:]
```
并计算交叉熵损失。

**6 自回归生成**
- 推理时根据最后一个位置的 Logits 选择下一个 Token。
- 可使用 Temperature、Top-K、Top-P 和重复惩罚控制生成结果。
- 新 Token 被追加到序列末尾，然后继续下一轮生成，直到遇到 EOS 或达到最大生成长度。

## 2.2 整体架构组装代码实现

### 2.2.1 MiniMindModel
**MiniMind 主干网络，负责 Embedding、堆叠多个 Block、RoPE 和 KV Cache**
```texte
input_ids -> 词向量emdedding
        |
 transformer layers  --------- 包括attention层和FFN
        ｜
最后得到：hidden_state,更新的KV cache
```


MiniMindModel 是不包含词表输出层的主干网络，主要负责：
- 把 input_ids 转换为词向量；
- 预计算并管理 RoPE 位置编码；
- 依次执行多个 MiniMindBlock；
- 管理每一层的 KV Cache；
- 汇总 MoE 辅助损失；
- 输出最终隐藏状态。

In [ ]:
class MiniMindModel(nn.Module):
    """MiniMind 主干模型，负责词嵌入、Transformer 层堆叠、RoPE 和 KV Cache。"""

    def __init__(self, config: MiniMindConfig):
        super().__init__()

        # 保存模型配置，供前向传播和位置编码重建使用
        self.config = config
        # 保存词表大小（因为最终的hidden_size要映射到词表上）和 Transformer 层数
        self.vocab_size, self.num_hidden_layers = config.vocab_size, config.num_hidden_layers
        # 将 Token ID 映射为 hidden_size 维的词向量
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        # 对词嵌入应用 Dropout，为了
        self.dropout = nn.Dropout(config.dropout)
        # 创建并堆叠多个 Transformer Block
        self.layers = nn.ModuleList([MiniMindBlock(l, config) for l in range(self.num_hidden_layers)])
        # 所有 Transformer 层执行完成后的最终归一化
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

        # 提前计算所有位置对应的 RoPE 正弦和余弦值（在每一个 Transformer Block 中重复使用）
        freqs_cos, freqs_sin = precompute_freqs_cis(
            dim=config.head_dim,
            end=config.max_position_embeddings,
            rope_base=config.rope_theta,
            rope_scaling=config.rope_scaling
        )

        # 注册为模型缓冲区，使其可以跟随模型移动到 CPU/GPU
        # persistent=False 表示不将其保存到模型权重文件中
        self.register_buffer("freqs_cos", freqs_cos, persistent=False)
        self.register_buffer("freqs_sin", freqs_sin, persistent=False)

    def forward(self, input_ids, attention_mask=None, past_key_values=None, use_cache=False, **kwargs):
        # input_ids 的形状为 [batch_size, sequence_length]
        batch_size, seq_length = input_ids.shape

        # 当前实现不直接使用 transformers 新版 Cache 对象
        # 遇到新版缓存格式时，将其重置为空缓存
        if hasattr(past_key_values, "layers"):
            past_key_values = None

        # 没有历史缓存时，为每个 Transformer 层创建一个空缓存
        past_key_values = past_key_values or [None] * len(self.layers)

        # 如果存在 KV Cache，则缓存中的 Key 长度就是本次输入的起始位置
        start_pos = past_key_values[0][0].shape[1] if past_key_values[0] is not None else 0

        # Token ID → 词向量，并应用 Dropout
        #Dropout--是一种正则化：训练时随机把一部分 embedding 维度置零，防止模型过度依赖某些固定的词向量特征。
        # [B, S] → [B, S, hidden_size]
        hidden_states = self.dropout(self.embed_tokens(input_ids))

        # transformers 5.x 使用 meta device 初始化时可能丢失非持久化缓冲区
        # 检测到 RoPE 缓冲区无效后重新计算
        if self.freqs_cos[0, 0] == 0:
            freqs_cos, freqs_sin = precompute_freqs_cis(
                dim=self.config.head_dim,
                end=self.config.max_position_embeddings,
                rope_base=self.config.rope_theta,
                rope_scaling=self.config.rope_scaling
            )

            # 将重新计算的 RoPE 数据移动到隐藏状态所在设备
            self.freqs_cos = freqs_cos.to(hidden_states.device)
            self.freqs_sin = freqs_sin.to(hidden_states.device)

        # 根据当前起始位置和序列长度截取出本次需要的 RoPE 数据
        position_embeddings = (
            self.freqs_cos[start_pos:start_pos + seq_length],
            self.freqs_sin[start_pos:start_pos + seq_length]
        )

        # 保存每个 Transformer 层生成的 KV Cache
        presents = []

        # 依次执行所有 Transformer Block
        for layer, past_key_value in zip(self.layers, past_key_values):
            hidden_states, present = layer(
                hidden_states,
                position_embeddings,
                past_key_value=past_key_value,
                use_cache=use_cache,
                attention_mask=attention_mask
            )

            # 保存当前层的新 KV Cache，每层的KV cache不断加入到presents列表中，最终返回给调用者
            presents.append(present)

        # 对所有 Transformer 层的最终输出进行归一化
        hidden_states = self.norm(hidden_states)

        # 如果使用 MoE，则汇总每一层的专家负载均衡辅助损失
        # 普通 FFN 模型没有该损失，结果为 0
        aux_loss = sum(
            [layer.mlp.aux_loss for layer in self.layers if isinstance(layer.mlp, MOEFeedForward)],
            hidden_states.new_zeros(1).squeeze()
        )

        # 返回隐藏状态、每层 KV Cache 和 MoE 辅助损失
        return hidden_states, presents, aux_loss

注意，回看MiniMind架构图，到现在为止，我们搭建的网络仅到RMSNorm层，后面的Linear和SoftMax还没有添加。

接下来需要进一步封装一个MiniMindForCausalLM类，这是为了更好地应用该模型于**因果语言建模任务（Causal Language Modeling）**，并增强其在推理、训练、生成等任务中的灵活性与兼容性。

### 2.2.2 MiniMindForCausalLM
**完整的语言模型封装，因果语言模型外层**，负责
- 输出词表概率
- 计算训练损失
- 自回归生成

因为虽然MiniMindModel已经实现了Transformer主干（包括嵌入层、注意力模块等核心组件），但它只负责将输入的token ID编码为hidden states，属于“纯 backbone”模块。

而MiniMindForCausalLM是一个“任务级封装”，它在主干模型基础上加上了输出层（language modeling head，lm_head）和统一的输出结构，用于直接进行token-level的预测。

MiniMindForCausalLM 在主干网络外增加了语言模型输出层，负责：
- 将隐藏状态映射为词表 logits；
- 计算 Next Token Prediction 损失；
- 接入 Hugging Face 模型和生成接口；
- 使用 Temperature、Top-K、Top-P 等策略生成文本；
- 利用 KV Cache 加速自回归生成。

In [ ]:
class MiniMindForCausalLM(PreTrainedModel, GenerationMixin):
    """MiniMind 因果语言模型，负责词表预测、训练损失和文本生成。"""

    # 指定当前模型使用的配置类
    config_class = MiniMindConfig

    # 告诉 Hugging Face，LM Head 和输入 Embedding 可以共享权重
    _tied_weights_keys = {"lm_head.weight": "model.embed_tokens.weight"}

    def __init__(self, config: MiniMindConfig = None):
        # 没有传入配置时使用默认配置
        self.config = config or MiniMindConfig()
        # 初始化 Hugging Face PreTrainedModel
        super().__init__(self.config)

        # 实例化一个minimind模型，负责词嵌入、Transformer 层堆叠、RoPE 和 KV Cache
        self.model = MiniMindModel(self.config)

        # 将隐藏状态映射为词表中每个 Token 的预测分数
        # [B, S, hidden_size] → [B, S, vocab_size]
        self.lm_head = nn.Linear(self.config.hidden_size, self.config.vocab_size, bias=False)

        # 配置允许时，让输入 Embedding 和输出 LM Head 共享权重
        if self.config.tie_word_embeddings:
            self.model.embed_tokens.weight = self.lm_head.weight

        # 按照 Hugging Face 规范完成参数初始化
        self.post_init()

    def forward(self, input_ids, attention_mask=None, past_key_values=None, use_cache=False, logits_to_keep=0, labels=None, **kwargs):
        # 通过主干网络得到隐藏状态、KV Cache 和 MoE 辅助损失
        hidden_states, past_key_values, aux_loss = self.model(
            input_ids,
            attention_mask,
            past_key_values,
            use_cache,
            **kwargs
        )

        # logits_to_keep=0 时保留全部位置
        # 生成时可以只保留最后几个位置，减少 LM Head 计算量
        slice_indices = slice(-logits_to_keep, None) if isinstance(logits_to_keep, int) else logits_to_keep

        # 将隐藏状态转换为词表预测分数
        logits = self.lm_head(hidden_states[:, slice_indices, :])

        # 没有 labels 时只进行预测，不计算训练损失
        loss = None

        if labels is not None:
            # 自回归语言模型使用前一个 Token 预测后一个 Token
            #
            # logits: 去掉最后一个位置
            # labels: 去掉第一个位置
            x = logits[..., :-1, :].contiguous()
            y = labels[..., 1:].contiguous()

            # 将序列展平后计算交叉熵（包含softmax）
            # label=-100 的位置不会参与损失计算
            loss = F.cross_entropy(
                x.view(-1, x.size(-1)),
                y.view(-1),
                ignore_index=-100
            )

        # 使用 Hugging Face 标准输出结构返回结果
        return MoeCausalLMOutputWithPast(
            loss=loss,
            aux_loss=aux_loss,
            logits=logits,
            past_key_values=past_key_values,
            hidden_states=hidden_states
        )


In [ ]:
# 推理模式：关闭梯度计算，降低显存占用
    @torch.inference_mode()
    def generate(self, inputs=None, attention_mask=None, max_new_tokens=8192, temperature=0.85, top_p=0.85, top_k=50, eos_token_id=2, streamer=None, use_cache=True, num_return_sequences=1, do_sample=True, repetition_penalty=1.0, **kwargs):
        # 同一个输入复制多份，用于一次生成多个候选结果
        input_ids = kwargs.pop("input_ids", inputs).repeat(num_return_sequences, 1)

        # attention_mask 也需要按生成结果数量复制
        attention_mask = attention_mask.repeat(num_return_sequences, 1) if attention_mask is not None else None

        # 获取已有 KV Cache，续写场景中可以直接复用
        past_key_values = kwargs.pop("past_key_values", None)

        # 记录每个生成序列是否已经遇到 EOS
        finished = torch.zeros(input_ids.shape[0], dtype=torch.bool, device=input_ids.device)

        # 将原始输入发送给流式输出器
        if streamer:
            streamer.put(input_ids.cpu())

        # 每轮生成一个新 Token
        for _ in range(max_new_tokens):
            # 已缓存的历史序列长度
            past_len = past_key_values[0][0].shape[1] if past_key_values else 0

            # 使用缓存时只需输入尚未计算过的新 Token
            outputs = self.forward(
                input_ids[:, past_len:],
                attention_mask,
                past_key_values,
                use_cache=use_cache,
                **kwargs
            )

            # 新生成一个 Token 后，注意力掩码也增加一个有效位置
            attention_mask = torch.cat(
                [attention_mask, attention_mask.new_ones(attention_mask.shape[0], 1)],
                dim=-1
            ) if attention_mask is not None else None

            # 只取最后一个位置的预测结果，并用温度调节概率分布
            logits = outputs.logits[:, -1, :] / temperature

            # 对已经出现过的 Token 施加重复惩罚
            if repetition_penalty != 1.0:
                for i in range(input_ids.shape[0]):
                    seen = torch.unique(input_ids[i])
                    score = logits[i, seen]
                    logits[i, seen] = torch.where(
                        score > 0,
                        score / repetition_penalty,
                        score * repetition_penalty
                    )

            # Top-K：仅保留概率最高的 K 个候选 Token
            if top_k > 0:
                threshold = torch.topk(logits, top_k)[0][..., -1, None]
                logits[logits < threshold] = -float("inf")

            # Top-P：仅保留累计概率不超过 top_p 的候选 Token
            if top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                mask = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1) > top_p

                # 至少保留概率最高的一个 Token
                mask[..., 1:] = mask[..., :-1].clone()
                mask[..., 0] = False

                # 将排序后的掩码恢复到原始词表顺序
                logits[mask.scatter(1, sorted_indices, mask)] = -float("inf")

            # do_sample=True 时按概率采样，否则直接选择概率最大的 Token
            next_token = torch.multinomial(torch.softmax(logits, dim=-1), num_samples=1) if do_sample else torch.argmax(logits, dim=-1, keepdim=True)

            # 已完成的序列后续始终填充 EOS
            if eos_token_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(-1),
                    next_token.new_full((next_token.shape[0], 1), eos_token_id),
                    next_token
                )

            # 将新 Token 添加到已有序列末尾
            input_ids = torch.cat([input_ids, next_token], dim=-1)

            # 保存最新 KV Cache，下一轮不再重复计算历史 Token
            past_key_values = outputs.past_key_values if use_cache else None

            # 流式输出本轮生成的 Token
            if streamer:
                streamer.put(next_token.cpu())

            # 更新各序列的结束状态
            if eos_token_id is not None:
                finished |= next_token.squeeze(-1).eq(eos_token_id)

                # 所有序列都已结束时提前退出
                if finished.all():
                    break

        # 通知流式输出器生成结束
        if streamer:
            streamer.end()

        # 某些续写场景需要同时返回生成结果和 KV Cache
        if kwargs.get("return_kv"):
            return {
                "generated_ids": input_ids,
                "past_kv": past_key_values
            }

        # 默认只返回生成后的完整 Token 序列
        return input_ids